<a href="https://colab.research.google.com/github/mahidodiya/AI-ML/blob/main/rank_test1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install -q transformers accelerate librosa soundfile pillow av sentencepiece
!apt-get -qq install -y ffmpeg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 22.1 MB/s eta 0:00:00


In [5]:
import os
import torch
import librosa
from PIL import Image
from transformers import (
    pipeline, AutoTokenizer, AutoModelForSeq2SeqLM,
    BlipProcessor, BlipForConditionalGeneration,
)

In [6]:
class MultimodalContentDescriber:
    def __init__(self):
        self.use_gpu = torch.cuda.is_available()
        self.device = 0 if self.use_gpu else -1
        self.torch_device = torch.device("cuda" if self.use_gpu else "cpu")
        print(f"🚀 Initializing pipelines on: {'GPU' if self.use_gpu else 'CPU'}")

        # 1. Image captioning (BLIP), loaded directly because the
        #    "image-to-text" pipeline was removed in transformers v5
        self.blip_processor = BlipProcessor.from_pretrained(
            "Salesforce/blip-image-captioning-base"
        )
        self.blip_model = BlipForConditionalGeneration.from_pretrained(
            "Salesforce/blip-image-captioning-base"
        ).to(self.torch_device)
        self.blip_model.eval()

        # 2. Text summarization (BART), loaded directly for compatibility
        self.sum_tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
        self.sum_model = AutoModelForSeq2SeqLM.from_pretrained(
            "facebook/bart-large-cnn"
        ).to(self.torch_device)
        self.sum_model.eval()

        # 3. Speech recognition (Whisper), chunked so audio > 30s isn't cut off
        self.audio_pipeline = pipeline(
            "automatic-speech-recognition",
            model="openai/whisper-tiny",
            chunk_length_s=30,
            device=self.device,
        )

        # 4. Video action classification (VideoMAE), needs the `av` package
        self.video_pipeline = pipeline(
            "video-classification",
            model="MCG-NJU/videomae-base-short-finetuned-kinetics",
            device=self.device,
        )

    @staticmethod
    def _check_file(path):
        if not os.path.isfile(path):
            raise FileNotFoundError(f"File not found: {path}")

    def describe_image(self, image_path):
        self._check_file(image_path)
        image = Image.open(image_path).convert("RGB")
        inputs = self.blip_processor(images=image, return_tensors="pt").to(self.torch_device)
        with torch.no_grad():
            ids = self.blip_model.generate(**inputs, max_new_tokens=40)
        caption = self.blip_processor.decode(ids[0], skip_special_tokens=True)
        return f"🖼️ Image Description: {caption}"

    def describe_text(self, text_content):
        text_content = (text_content or "").strip()
        if not text_content:
            return "📝 Text Content: (empty)"
        if len(text_content.split()) < 30:
            return f'📝 Text Content: "{text_content}"'

        inputs = self.sum_tokenizer(
            text_content, return_tensors="pt", truncation=True, max_length=1024
        ).to(self.torch_device)

        input_len = inputs["input_ids"].shape[1]
        max_len = max(20, min(60, input_len // 2))
        min_len = min(10, max_len - 5)

        with torch.no_grad():
            ids = self.sum_model.generate(
                **inputs,
                max_length=max_len,
                min_length=min_len,
                num_beams=4,
                do_sample=False,
                early_stopping=True,
            )
        summary = self.sum_tokenizer.decode(ids[0], skip_special_tokens=True)
        return f"📝 Text Summary: {summary}"

    def describe_audio(self, audio_path):
        self._check_file(audio_path)
        speech, rate = librosa.load(audio_path, sr=16000, mono=True)
        result = self.audio_pipeline({"raw": speech, "sampling_rate": rate})
        return f'🔊 Audio Transcript: "{result["text"].strip()}"'

    def describe_video(self, video_path):
        self._check_file(video_path)
        result = self.video_pipeline(video_path, top_k=2)
        actions = [f"{r['label']} (conf: {r['score']:.2f})" for r in result]
        return f"🎥 Video Action Analysis: Detected {', '.join(actions)}"

In [7]:
describer = MultimodalContentDescriber()

🚀 Initializing pipelines on: GPU


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/162 [00:00<?, ?it/s]

[transformers] VideoMAEForVideoClassification LOAD REPORT from: MCG-NJU/videomae-base-short-finetuned-kinetics
Key                                                            | Status     | 
---------------------------------------------------------------+------------+-
videomae.encoder.layer.{0...11}.attention.attention.v_bias     | UNEXPECTED | 
videomae.encoder.layer.{0...11}.attention.attention.q_bias     | UNEXPECTED | 
videomae.encoder.layer.{0...11}.attention.attention.query.bias | MISSING    | 
videomae.encoder.layer.{0...11}.attention.attention.key.bias   | MISSING    | 
videomae.encoder.layer.{0...11}.attention.attention.value.bias | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


preprocessor_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

In [19]:
sample_text = (
    "Hugging Face is a platform that allows users to share machine learning "
    "models and datasets. It serves as a central hub for open-source AI "
    "deployment globally, and it hosts thousands of community-contributed "
    "models across text, vision, and audio."
)
print(describer.describe_text(sample_text))

📝 Text Summary: Hugging Face is a platform that allows users to share machine learning models and datasets. It serves as a central


In [14]:
print(describer.describe_image("1_safe.png"))
print(describer.describe_image("image.jpg"))
print(describer.describe_image("1_cat.png"))
print(describer.describe_image("9_test3.jpeg"))
print(describer.describe_image("11_test4.png"))

🖼️ Image Description: the streets of paris
🖼️ Image Description: a man and woman dancing
🖼️ Image Description: a cat sitting on a wall
🖼️ Image Description: a group of people walking down the street
🖼️ Image Description: a couple walking across a crosswalk in tokyo


In [18]:
print(describer.describe_audio("angry_01.wav"))

🔊 Audio Transcript: "Excuse me? You fucking put me across the street? How fucking day you make me mad at you? You can't write with him. Fuck it. Thank you. Fucker. Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Cori! Co

In [21]:
print(describer.describe_video("dog.mp4"))

🎥 Video Action Analysis: Detected walking the dog (conf: 0.31), catching fish (conf: 0.08)


In [20]:
print(describer.describe_video("japanese-people.mp4"))

🎥 Video Action Analysis: Detected waiting in line (conf: 0.32), marching (conf: 0.09)


In [ ]:
print(describer.describe_video("japanese-people.mp4"))